## MAPA DE MOVIMENTAÇÃO CÍVEL ##

#### CARREGA AS BIBLIOTECAS E INSTALA AS BIBLIOTECAS FALTANTES ####

In [0]:
pip install xlsxwriter

In [0]:
pip install pyxlsb 

In [0]:
pip install xlwings

In [0]:
import pandas as pd
import os
import openpyxl 
import xlwings
import xlsxwriter
import shutil
import pyxlsb
import shutil
import numpy as np
import glob

#### CARREGA AS BASES ####

In [0]:
dbutils.widgets.text("Mes_Fechamento", "", "Base Fechamento Mensal")
Mes_Fechamento = dbutils.widgets.get("Mes_Fechamento")

In [0]:
meses_do_ano = [
    "JANEIRO", "FEVEREIRO", "MARÇO", "ABRIL", "MAIO", "JUNHO",
    "JULHO", "AGOSTO", "SETEMBRO", "OUTUBRO", "NOVEMBRO", "DEZEMBRO"
]

dbutils.widgets.dropdown(
    name="Mes_Extenso", 
    defaultValue="OUTUBRO", 
    choices=meses_do_ano, 
    label="Selecione o Mês por Extenso"
)

Meses = dbutils.widgets.get("Mes_Extenso")

In [0]:
#Carrega a base do fechamento do mês
base_fechamento_mensal = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/CÍVEL/BASE_FECHAMENTO_MENSAL/{Mes_Fechamento} Fechamento Cível.xlsx'
#MAPA DE CALOR MES PASSADO PARA VALIDAÇÃO
mapa_movi_mes_passado = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/CÍVEL/MAPA MOVIMENTAÇÃO MES ANTERIOR/Mapa de Movimentação Cível - {Meses} 2025.xlsx'
#TEMPLATE DO MAPA DE MOVIMENTAÇÃO
template = f'/Volumes/databox/juridico_comum/arquivos/MAPAS_MOVIMENTAÇÃO/INPUT/TEMPLATE/Mapa de Movimentação Template.xlsx'

In [0]:
df_fechamento = pd.read_excel(
    base_fechamento_mensal,
    sheet_name= 'Base',
    header=1
)

df_procv = pd.read_excel(
    base_fechamento_mensal,
    sheet_name= 'DEPARA_MAPA_MOVI',
    header=0
)

df_map_past = pd.read_excel(
    mapa_movi_mes_passado,
    sheet_name= 'Mapa de Movimentação',
    header=5
)

df_template = pd.read_excel(
    template,
    sheet_name= 'Mapa de Movimentação',
    header=7
)


### Começa o tratamento, primeiro passo é criar a coluna de Classificação dos valores provaveis e remotos ###

In [0]:
#CRIA A COLUNA CLASSIFICAÇÃO COM VALORES DE PROVAVEL OU REMOTO
cond_1 =  df_fechamento['PROVISÃO TOTAL (M)'] != 0
df_fechamento['Classificação'] = np.where(cond_1,'Provavel','remoto')

In [0]:
# Realziar o merge (procv) e traz a coluna com os nomes resumidos 

df_procv_res = df_procv[['EMPRESA (BASE FECHAMENTO)','EMPRESA (MAPA_NOVO)']]

df_fechamento_completo = pd.merge(
    df_fechamento,
    df_procv_res,
    left_on='EMPRESA (M)',
    right_on='EMPRESA (BASE FECHAMENTO)',
    how='left'
).drop('EMPRESA (BASE FECHAMENTO)',axis=1)

In [0]:
# Lista de empresas que possuem as variantes "ESPECIAL" e "MASSA"
empresas_especiais = ['BARTIRA', 'GLOBEX', 'GCBON', 'GCB']

# Definir as CONDIÇÕES para a mudança
conditions = [
    # Regra 1: Se for "CÍVEL ESTRATÉGICO" E ESTIVER NA LISTA
    (df_fechamento_completo['ÁREA DO DIREITO'] == 'CÍVEL ESTRATÉGICO') & 
    (df_fechamento_completo['EMPRESA (MAPA_NOVO)'].isin(empresas_especiais)),
    
    # Regra 2: Se for "TRABALHISTA" E ESTIVER NA LISTA
    (df_fechamento_completo['ÁREA DO DIREITO'] == 'CÍVEL MASSA') & 
    (df_fechamento_completo['EMPRESA (MAPA_NOVO)'].isin(empresas_especiais))
]

In [0]:
# Definir os RESULTADOS (escolhas) para cada condição
choices = [
    df_fechamento_completo['EMPRESA (MAPA_NOVO)'] + " ESPECIAL",    # Resultado Regra 1
    df_fechamento_completo['EMPRESA (MAPA_NOVO)'] + " MASSA"  # Resultado Regra 2
]

In [0]:
# Criamos uma nova coluna. 
df_fechamento_completo['EMPRESA_TEMPLATE'] = np.select(
    conditions, 
    choices, 
    default=df_fechamento_completo['EMPRESA (MAPA_NOVO)'] # O que fazer se NENHUMA condição bater
)

In [0]:
print("--- DataFrame Após Aplicar as Regras ---")
print(df_fechamento_completo[['ÁREA DO DIREITO', 'EMPRESA (MAPA_NOVO)', 'EMPRESA_TEMPLATE']], '\n')

In [0]:
# 1. Criar a tabela dinâmica APENAS para 'provavel'
df_provavel = df_fechamento_completo[df_fechamento_completo['Classificação'] == 'Provavel']

tab_provavel = pd.pivot_table(
    df_provavel, 
    index=['ÁREA DO DIREITO', 'EMPRESA_TEMPLATE'],
    values=['NOVOS','ESTOQUE'],
    aggfunc='sum',
    margins=False
)

In [0]:
# 2. Criar a tabela dinâmica APENAS para 'remoto'
df_remoto = df_fechamento_completo[df_fechamento_completo['Classificação'] == 'remoto']

tab_remoto = pd.pivot_table(
    df_remoto, 
    index=['ÁREA DO DIREITO', 'EMPRESA_TEMPLATE'],
    values=['NOVOS','ESTOQUE'],
    aggfunc='sum',
    margins=False
)

In [0]:
# tab_provavel e tab_remoto
print("--- Tabela Provável ---")
print(tab_provavel)
print("\n--- Tabela Remoto ---")
print(tab_remoto)

### Monta o Provável ###

In [0]:
# Transforma as pivot em df para realizar o merge
df_provavel = tab_provavel.reset_index()
df_remoto = tab_remoto.reset_index()

In [0]:
# 1. Encontre a linha 'ASAP' que é 'CÍVEL MASSA COLETIVO'
condicao_asap = (df_provavel['EMPRESA_TEMPLATE'] == 'ASAP') & \
                (df_provavel['ÁREA DO DIREITO'] == 'CÍVEL ESTRATÉGICO')

# 2. Muda a 'Área do Direito' dessa linha para 'CÍVEL MASSA'
#    Agora, as linhas 0 e 8 têm as mesmas chaves: ('CÍVEL MASSA', 'ASAP')
df_provavel.loc[condicao_asap, 'ÁREA DO DIREITO'] = 'CÍVEL MASSA'

# 3. Agora, agrupe e some.
#    O GCB (linhas 4 e 9) não será afetado, pois suas chaves continuam diferentes.
df_provavel_agrupado = df_provavel.groupby(
    ['ÁREA DO DIREITO', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_provavel =  df_provavel_agrupado

In [0]:
df_tp_fin = df_template[['EMPRESA']]

In [0]:
df_map_past.head()

In [0]:
#REALIZA O MERGE PARA TRAZER O SALDO FINAL DO MÊS ANTERIOR QUE SERÁ NOSSO MÊS INICIAL DO MÊS ATUAL

chave_merge = 'EMPRESA'
coluna = 'Saldo Final.1'
 
df_map_past= df_map_past.drop_duplicates(subset=[chave_merge])

df_mes_pass = df_map_past[[chave_merge, coluna]]

df_final = pd.merge(
    df_tp_fin,
    df_mes_pass,  
    on=chave_merge, 
    how='left'
)

In [0]:
df_final = df_final.rename(columns={'Saldo Final.1': 'Provável Sdo inicial'})

# 1 trocar nan por 0
df_final['Provável Sdo inicial'] = df_final['Provável Sdo inicial'].fillna(0)

# 2 Converter float para int
df_final['Provável Sdo inicial'] = df_final['Provável Sdo inicial'].astype(int)

df_final.head(20)

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'NOVOS'

# 2. Criar o subset (isto está correto)
df_prov_merge = df_provavel[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final = pd.merge(
    df_final,
    df_prov_merge,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final = df_final.drop(chave_direita, axis=1)

In [0]:
df_final = df_final.rename(columns={'NOVOS': 'Novos'})

# 1 trocar nan por 0
df_final['Novos'] = df_final['Novos'].fillna(0)

# 2 Converter float para int
df_final['Novos'] = df_final['Novos'].astype(int)

df_drop = df_final.drop_duplicates(subset=['EMPRESA'], keep = 'first')

df_final = df_drop

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'ESTOQUE'

# 2. Criar o subset (isto está correto)
df_prov_merge1 = df_provavel[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final = pd.merge(
    df_final,
    df_prov_merge1,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final = df_final.drop(chave_direita, axis=1)

In [0]:
# 1 trocar nan por 0
df_final['ESTOQUE'] = df_final['ESTOQUE'].fillna(0)

# 2 Converter float para int
df_final['ESTOQUE'] = df_final['ESTOQUE'].astype(int)

In [0]:
# Bloco 34 Corrigido: Lógica para df_final (Provável)

# 1. Calcule o Saldo Teórico e a Diferença
df_final['Total_Teorico'] = df_final['Provável Sdo inicial'] + df_final['Novos']
df_final['Diferenca'] = df_final['Total_Teorico'] - df_final['ESTOQUE']

# 2. Defina as Condições
# Se a diferença for positiva ou zero, significa que houve mais casos que o estoque final, então o excedente é Encerrado.
cond_encerrado = df_final['Diferenca'] >= 0 

# Se a diferença for negativa, significa que faltaram casos para atingir o estoque final, então essa falta é uma Adição/Ajuste (resultado).
# Vamos usar o valor absoluto da Diferenca para atribuir a Adição/Ajuste.
valor_ajuste = df_final['Diferenca'].abs().astype(int)

In [0]:
# 3. ATUALIZAR A COLUNA 'resultado'
# Se cond_encerrado for FALSE (ou seja, a Diferenca é negativa), a Adição/Ajuste recebe o valor absoluto da Diferenca. Caso contrário, é 0.
df_final['resultado'] = np.where(
    ~cond_encerrado,    # Usamos o operador ~ (NOT) para inverter a condição. Se Diferenca < 0...
    valor_ajuste,       # ... então o Ajuste é o valor absoluto (ex: 1 para ASAP)
    0                   # ... caso contrário, é zero.
)

# 4. ATUALIZAR A COLUNA 'ENCERRADO'
# Se cond_encerrado for TRUE (ou seja, a Diferenca é positiva ou zero), o Encerrado é o valor positivo da Diferenca. Caso contrário, é 0.
df_final['ENCERRADO'] = np.where(
    cond_encerrado,      # Se Diferenca >= 0...
    df_final['Diferenca'].astype(int), # ... o Encerrado é o valor da Diferença (ex: 3 para BARTIRA MASSA)
    0                    # ... caso contrário, é zero.
)

# 5. Remoção das colunas auxiliares (opcional, mas recomendado)
df_final = df_final.drop(['Total_Teorico', 'Diferenca'], axis=1)

In [0]:
df_final.head(20)

In [0]:
df_final = df_final.rename(columns={'resultado': 'Adição'})

In [0]:
#df_final = df_final.drop(['ESTOQUE'], axis=1)
#df_final = df_final.drop(['Novos'], axis=1)

In [0]:
df_final.head()

### MONTA OS VALORES REMOTOS ###

In [0]:
# 1. Encontre a linha 'ASAP' que é 'TRABALHISTA COLETIVO'
condicao_asap = (df_remoto['EMPRESA_TEMPLATE'] == 'ASAP') & \
                (df_remoto['ÁREA DO DIREITO'] == 'CÍVEL ESTRATÉGICO')

# 2. Mude a 'Área do Direito' dessa linha para 'TRABALHISTA'
#    Agora, as linhas 0 e 8 têm as mesmas chaves: ('TRABALHISTA', 'ASAP')
df_remoto.loc[condicao_asap, 'ÁREA DO DIREITO'] = 'CÍVEL MASSA'

# 3.
#    O groupby vai somar as linhas 0 e 8 automaticamente.
#    O GCB (linhas 4 e 9) não será afetado, pois suas chaves continuam diferentes.
df_remoto_agrupado = df_remoto.groupby(
    ['ÁREA DO DIREITO', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_remoto =  df_remoto_agrupado

In [0]:
# 1. Encontre TODAS as linhas que são 'CÍVEL ESTRATÉGICO'
condicao_coletivo = (df_remoto['ÁREA DO DIREITO'] == 'CÍVEL ESTRATÉGICO')

# 2. Mude a 'Área do Direito' dessas linhas para 'TRABALHISTA'
#    Isso vai fazer com que ASAP, GCBHUB, etc., fiquem com a mesma chave.
df_remoto.loc[condicao_coletivo, 'ÁREA DO DIREITO'] = 'CÍVEL MASSA'

# 3. Agora, agrupe e some pelas chaves
#    Isso irá somar automaticamente ASAP, GCBHUB e qualquer outra empresa.
df_remoto_agrupado = df_remoto.groupby(
    ['ÁREA DO DIREITO', 'EMPRESA_TEMPLATE']
).sum().reset_index()

df_remoto =  df_remoto_agrupado

In [0]:
# Separa a coluna EMPRESA para criar o novo dataframe 
df_tp_fin1 = df_template[['EMPRESA']]

In [0]:
#Realiza o merge para trazer o valor final do mes anterior para se tornar o mes atual
chave_merge_remoto = 'EMPRESA'
coluna = 'Saldo Final.3'
 
df_map_past1= df_map_past.drop_duplicates(subset=[chave_merge_remoto])

df_mes_pass_remoto = df_map_past1[[chave_merge_remoto, coluna]]

df_final_remoto = pd.merge(
    df_tp_fin,
    df_mes_pass_remoto,  
    on=chave_merge, 
    how='left'
)

In [0]:
df_final_remoto = df_final_remoto.rename(columns={'Saldo Final.3': 'Remoto Sdo inicial'})

# 1 trocar nan por 0
df_final_remoto['Remoto Sdo inicial'] = df_final_remoto['Remoto Sdo inicial'].fillna(0)

# 2 Converter float para int
df_final_remoto['Remoto Sdo inicial'] = df_final_remoto['Remoto Sdo inicial'].astype(int)

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'NOVOS'

# 2. Criar o subset (isto está correto)
df_remoto_merge = df_remoto[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final_remoto = pd.merge(
    df_final_remoto,
    df_remoto_merge,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final_remoto = df_final_remoto.drop(chave_direita, axis=1)

In [0]:
#Converte os numeros de float para int
df_final_remoto = df_final_remoto.rename(columns={'NOVOS': 'Novos'})

# 1 trocar nan por 0
df_final_remoto['Novos'] = df_final_remoto['Novos'].fillna(0)

# 2 Converter float para int
df_final_remoto['Novos'] = df_final_remoto['Novos'].astype(int)

df_drop_remoto = df_final_remoto.drop_duplicates(subset=['EMPRESA'], keep = 'first')

df_final_remoto = df_drop_remoto

In [0]:
chave_direita = 'EMPRESA_TEMPLATE' # Chave no df_provavel
chave_esquerda = 'EMPRESA'        # Chave no df_final
coluna_valor = 'ESTOQUE'

# 2. Criar o subset (isto está correto)
df_remoto_merge1 = df_remoto[[chave_direita, coluna_valor]]

# 3. CORRIGIR O MERGE
df_final_remoto = pd.merge(
    df_final_remoto,
    df_remoto_merge1,
    left_on=chave_esquerda,     # Use 'EMPRESA' do df_final
    right_on=chave_direita,    # Use 'EMPRESA_TEMPLATE' do df_prov_merge
    how='left'
)

df_final_remoto = df_final_remoto.drop(chave_direita, axis=1)

In [0]:
# 1 trocar nan por 0
df_final_remoto['ESTOQUE'] = df_final_remoto['ESTOQUE'].fillna(0)

# 2 Converter float para int
df_final_remoto['ESTOQUE'] = df_final_remoto['ESTOQUE'].astype(int)

In [0]:
# Bloco 34 Corrigido: Lógica para df_final (Provável)

# 1. Calcule o Saldo Teórico e a Diferença
df_final_remoto['Total_Teorico'] = df_final_remoto['Remoto Sdo inicial'] + df_final_remoto['Novos']
df_final_remoto['Diferenca'] = df_final_remoto['Total_Teorico'] - df_final_remoto['ESTOQUE']

# 2. Defina as Condições
# Se a diferença for positiva ou zero, significa que houve mais casos que o estoque final, então o excedente é Encerrado.
cond_encerrado = df_final_remoto['Diferenca'] >= 0 

# Se a diferença for negativa, significa que faltaram casos para atingir o estoque final, então essa falta é uma Adição/Ajuste (resultado).
# Vamos usar o valor absoluto da Diferenca para atribuir a Adição/Ajuste.
valor_ajuste = df_final_remoto['Diferenca'].abs().astype(int)

In [0]:
# 3. ATUALIZAR A COLUNA 'resultado' (Adição/Ajuste)
# Se cond_encerrado for FALSE (ou seja, a Diferenca é negativa), a Adição/Ajuste recebe o valor absoluto da Diferenca. Caso contrário, é 0.
df_final_remoto['resultado'] = np.where(
    ~cond_encerrado,    # Usamos o operador ~ (NOT) para inverter a condição. Se Diferenca < 0...
    valor_ajuste,       # ... então o Ajuste é o valor absoluto (ex: 1 para ASAP)
    0                   # ... caso contrário, é zero.
)

# 4. ATUALIZAR A COLUNA 'ENCERRADO'
# Se cond_encerrado for TRUE (ou seja, a Diferenca é positiva ou zero), o Encerrado é o valor positivo da Diferenca. Caso contrário, é 0.
df_final_remoto['ENCERRADO'] = np.where(
    cond_encerrado,      # Se Diferenca >= 0...
    df_final_remoto['Diferenca'].astype(int), # ... o Encerrado é o valor da Diferença (ex: 3 para BARTIRA MASSA)
    0                    # ... caso contrário, é zero.
)

# 5. Remoção das colunas auxiliares (opcional, mas recomendado)
df_final_remoto = df_final_remoto.drop(['Total_Teorico', 'Diferenca'], axis=1)

In [0]:
df_final_remoto.head(20)

In [0]:
df_final_remoto = df_final_remoto.rename(columns={'resultado': 'Adição'})

In [0]:
df_final_remoto = df_final_remoto.drop(['ESTOQUE'], axis=1)
df_final_remoto = df_final_remoto.drop(['Novos'], axis=1)

In [0]:
df_final_remoto.head(20)

## Realiza a valorização por empresa ##